# qBraid Vault Challenge

Thirteen hidden **vault circuits** stand between you and a perfect score.

For each vault, your goal is to build a **Qiskit circuit** that, when appended **after** the hidden vault circuit, returns the combined system to the all-zeros state $|0\ldots0\rangle$ — in other words, find the circuit that *inverts* the vault. Every participant gets their own set of vaults, generated server-side just for them.

## The rules

| | Practice vault (0) | Scored vaults (1–12) |
|---|---|---|
| Probes | 50 | 20 each |
| Attacks | 50 | 20 each |
| Counts toward score | No | Yes |

- **`probe(vault_index, circuit)`** — runs the combined circuit for 200 shots and returns a measurement histogram. Reconnaissance only; never scored, but it spends a probe.
- **`attack(vault_index, circuit)`** — runs the combined circuit and returns a score:
  $$\text{score} = \text{rawScore} \times \text{costFactor}$$
  where **rawScore** is the probability of measuring all zeros — computed exactly, not sampled, so the same solution always scores the same — and **costFactor** $= \frac{4\,c}{4\,c + \Delta}$, with $c$ the number of two-qubit gates in the hidden vault circuit and $\Delta$ the **total** number of two-qubit gates in *your* circuit. Submit no entangling gates at all and costFactor is 1; every one you use costs a little.
- Your **total score** is the average of your best attack on vaults 1–12. Leaderboard ties rank by fewest scored attacks used — efficiency wins.
- The combined circuit (vault + yours) may use at most **20 qubits** and **10,000 operations**. Terminal measurements in your circuit are ignored (stripped before simulation); mid-circuit measurements are rejected as invalid.
- Requests are limited to **30 per minute** (probes + attacks combined). A submission that fails never costs you budget — invalid programs and 30-second timeouts are both refunded.

You are enrolled automatically on your first probe or attack — no registration needed.

## Setup

The client authenticates with your qBraid API key (already configured in qBraid Lab). Outside Lab, run `qbraid configure` first.

You write plain Qiskit — the client converts your circuit to OpenQASM 3 before sending it.

In [ ]:
# Confirm this notebook is running on the "Vault Challenge" kernel.
# If not, this prints exactly how to install the env, add its kernel, or switch to it.
from qbraid_kernel_check import check_vault_kernel
check_vault_kernel()

In [ ]:
# Dependencies are preinstalled in the qBraid "Vault Challenge" environment,
# so this is only needed if you're running the notebook OUTSIDE qBraid Lab.
# Uncomment to install locally:
# %pip install --quiet qbraid qiskit

In [ ]:
from qiskit import QuantumCircuit
from vault_client import VaultClient

client = VaultClient()
client.state()

## Warm-up: the practice vault

Vault 0 is the same for everyone and never scored — use it to learn the mechanics. Start by probing it with an *empty* circuit (same register, no gates) to see what state the vault leaves behind. Histogram keys are the big-endian decimal value of the measured bitstring: for 3 qubits, `"0"` = `000` and `"7"` = `111`.

In [ ]:
empty_probe = QuantumCircuit(3)   # a register, no gates

client.probe(0, empty_probe)

Roughly 50/50 between `"0"` and `"7"`... only $|000\rangle$ and $|111\rangle$ are ever measured. That signature should look familiar — it's a **GHZ state**: $\tfrac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$, prepared by an H on qubit 0 followed by CNOTs fanning out.

To unlock the vault, apply the inverse: undo the CNOTs (in reverse order), then undo the H.

In [ ]:
ghz_inverse = QuantumCircuit(3)
ghz_inverse.cx(0, 2)
ghz_inverse.cx(0, 1)
ghz_inverse.h(0)

client.attack(0, ghz_inverse)

A perfect `rawScore` of 1.0! But notice the final score is 0.8: the vault's cost base is 2 (its two CNOTs) and our inverse uses 2 two-qubit gates of its own, so `costFactor` $= \frac{4 \cdot 2}{4 \cdot 2 + 2} = \frac{8}{8+2} = 0.8$. Cheaper unlocks score higher — sometimes a *partial* inversion with fewer gates beats a perfect one.

Qiskit's own tools work here too: `circuit.inverse()` gives you the exact undo of any circuit you can guess, which is often the fastest way to test a hypothesis.

## The real vaults

Vaults 1–12 are yours alone. They come from three families, and within each family the locks get harder as the number climbs — but the families are different *kinds* of puzzle, so a higher number is not automatically a harder vault:

- **Vaults 1–4** — matrix-product-state circuits (6–12 qubits, 1–3 layers)
- **Vaults 5–8** — perturbed graph states (3–14 qubits). The graph structure is still there and worth recovering, but small single-qubit rotations mean its tell-tale parities are only *strongly biased*, never exactly deterministic — textbook stabilizer identification, which assumes exact determinism, will find nothing.
- **Vaults 9–12** — hardware-efficient ansätze (3–5 qubits, entangling layers)

Probe them, form a hypothesis about their structure, and attack.

In [ ]:
# Vault 5 is the smallest of the perturbed graph states — a good first target.
client.probe(5, QuantumCircuit(3))

## Tracking your progress

In [ ]:
state = client.state()
print(f"Total score: {state['totalAvgScore']:.3f}")
print(f"Attacks remaining per vault: {state['attacksRemaining']}")

In [ ]:
for row in client.leaderboard()[:10]:
    print(f"{row['rank']:>3}. {row['userName']:<24} {row['totalAvgScore']:.3f}  ({row['attacksUsed']} attacks)")

Good luck — and remember: fewer gates, higher scores. 🔓